# Increment 1: baselines and RoBERTa Tagalog fine-tuning (validation only)

**Rules this notebook enforces (Chapter 3, Sections 3.1 and 3.3.3):**
1. The **test split is dropped in Cell 1 and never used.** Everything here is tuned and compared on the **validation** split only. Do not add the test split to this notebook.
2. Positive class = malicious (smishing or spam). Smishing and spam recall are reported separately.
3. SMOTE is applied to **training embeddings only**; its effect is the difference from the class-weighted run, measured on validation.
4. Records seeds, config and results so the Results chapter can be reproduced.

**Before running:** Settings -> Accelerator: GPU. Settings -> Internet: ON (needed once to download the model). Add your PRIVATE dataset containing `corpus_kaggle.csv` (Add Data). Set `SMOKE = True` first to test the whole pipeline in a couple of minutes, then set it to `False`.

**Not tested before delivery:** the RoBERTa cells (5 to 7) could not be run where this notebook was written (no GPU, no network). Cells 1, 3 and 4 were run on a simulated corpus. Expect to fix small errors in cells 2 and 5 to 7.

In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files)

/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/robbieduardreyes []
/kaggle/input/datasets/robbieduardreyes/corpus-kaggle ['corpus_kaggle.csv']
/kaggle/input/datasets/robbieduardreyes/corpus-kaggle-for-thesis ['increment1_baselines_roberta.ipynb']


In [2]:
import os, glob, sys, json, math, random, copy, time, warnings
import numpy as np, pandas as pd
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------- CONFIG (edit here only) -----------------------------
SMOKE = False            # True = tiny dry run (200 train rows, 1 epoch, 1 seed)
SEEDS = [42, 43, 44]
MODEL_ID = "jcblaise/roberta-tagalog-base"   # VERIFY this id exists on the Hugging Face Hub first
MAX_LEN = 256            # set after reading the coverage table in Cell 2
EPOCHS = 4
LR = 2e-5
BATCH = 16
RUN_LINK_ABLATION = False    # True = also train with the [LINK] token removed (Section 3.1)
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
# -----------------------------------------------------------------------------------

cands = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print("CSV files under /kaggle/input:", cands)
path = os.environ.get("CORPUS_PATH")
if not path:
    for c in cands:
        cols = set(pd.read_csv(c, nrows=1).columns)
        if "text_norm" in cols or "sender" in cols:
            raise SystemExit(f"{c} is the FULL corpus (unmasked text). Delete that dataset and upload corpus_kaggle.csv instead.")
        if {"msg_id", "text_model", "label_bin", "split"} <= cols:
            path = c
            break
print("using:", path)
assert path, "corpus csv not found: use Add Data to attach your private dataset"
df = pd.read_csv(path, keep_default_na=False)
need = {"msg_id", "text_model", "has_url", "n_chars", "label3", "label_bin", "split"}
assert need <= set(df.columns), f"missing columns: {need - set(df.columns)}"
df["has_url"] = df["has_url"].astype(str).str.lower().eq("true")
df["label_bin"] = df["label_bin"].astype(int)
df["n_chars"] = df["n_chars"].astype(int)

# ---- THE TEST SPLIT IS SEALED: dropped here, never evaluated or inspected ----
df = df[df["split"] != "test"].reset_index(drop=True)
assert "test" not in set(df["split"])
tr = df[df.split == "train"].reset_index(drop=True)
va = df[df.split == "val"].reset_index(drop=True)
if SMOKE:
    tr = tr.sample(200, random_state=0).reset_index(drop=True)
    EPOCHS, SEEDS = 1, [42]
print(f"train {len(tr)} (malicious {tr.label_bin.mean():.1%}) | val {len(va)} (malicious {va.label_bin.mean():.1%})")
print("val smishing / spam / legit:", va.label3.value_counts().to_dict())
print("val malicious without a URL:", int(((va.label_bin == 1) & ~va.has_url).sum()))

CSV files under /kaggle/input: ['/kaggle/input/datasets/robbieduardreyes/corpus-kaggle/corpus_kaggle.csv']
using: /kaggle/input/datasets/robbieduardreyes/corpus-kaggle/corpus_kaggle.csv
train 1786 (malicious 61.8%) | val 381 (malicious 61.9%)
val smishing / spam / legit: {'spam': 174, 'legitimate': 145, 'smishing': 62}
val malicious without a URL: 20


In [3]:
# ---- Cell 2: token-length coverage (fills the [128/256] and [X]% placeholders) ----
from transformers import AutoTokenizer
PLACEHOLDERS = ["[LINK]", "[PHONE_NUM]", "[OTP]", "[AMOUNT]", "[REF]", "[EMAIL]"]

def make_tok():
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    # NOTE: this REPLACES any existing additional_special_tokens (RoBERTa has none by default).
    tok.add_special_tokens({"additional_special_tokens": PLACEHOLDERS})
    return tok

tok0 = make_tok()
lens = np.array([len(tok0(t, truncation=False)["input_ids"]) for t in df["text_model"]])
print("tokens per message (train+val): median", int(np.median(lens)),
      "| p95", int(np.percentile(lens, 95)), "| p99", int(np.percentile(lens, 99)), "| max", int(lens.max()))
for L in [64, 96, 128, 160, 192, 256, 384, 512]:
    print(f"  max_len {L:3d} covers {(lens <= L).mean():6.1%} of messages")
print("Chapter 1 / 3 placeholder: pick MAX_LEN, then report this coverage figure.")

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/472k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/272k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/804k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

tokens per message (train+val): median 41 | p95 123 | p99 166 | max 264
  max_len  64 covers  77.8% of messages
  max_len  96 covers  87.2% of messages
  max_len 128 covers  95.5% of messages
  max_len 160 covers  98.5% of messages
  max_len 192 covers  99.6% of messages
  max_len 256 covers 100.0% of messages
  max_len 384 covers 100.0% of messages
  max_len 512 covers 100.0% of messages
Chapter 1 / 3 placeholder: pick MAX_LEN, then report this coverage figure.


In [4]:
# ---- Cell 3: metrics and result bookkeeping ----
def prf(y, p):
    y, p = np.asarray(y), np.asarray(p)
    acc = accuracy_score(y, p)
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="binary", pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(acc=acc, prec=pr, rec=rc, f1=f1, tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def sub_recalls(d, p):
    p = np.asarray(p); out = {}
    for name, mask in [("recall_smishing", d.label3 == "smishing"),
                       ("recall_spam", d.label3 == "spam"),
                       ("recall_mal_no_url", (d.label_bin == 1) & ~d.has_url)]:
        m = mask.to_numpy()
        out[name] = float((p[m] == 1).mean()) if m.sum() else float("nan")
        out["n_" + name[7:]] = int(m.sum())
    return out

def boot_f1_ci(y, p, B=1000, seed=0):
    rng = np.random.default_rng(seed); y, p = np.asarray(y), np.asarray(p); f = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y)); f.append(prf(y[i], p[i])["f1"])
    return tuple(np.percentile(f, [2.5, 97.5]))

results = []
def record(name, seed, probs, d=None):
    """probs = P(malicious) per validation message (hard 0/1 predictions are fine for baselines)."""
    d = va if d is None else d
    probs = np.asarray(probs, dtype=float); pred = (probs >= 0.5).astype(int)
    r = dict(model=name, seed=seed, **prf(d.label_bin.values, pred), **sub_recalls(d, pred))
    lo, hi = boot_f1_ci(d.label_bin.values, pred); r["f1_ci_lo"], r["f1_ci_hi"] = lo, hi
    grid = np.arange(0.1, 0.91, 0.05)
    r["best_val_threshold_INFO_ONLY"] = float(grid[np.argmax([prf(d.label_bin.values, (probs >= t).astype(int))["f1"] for t in grid])])
    results.append(r)
    print(f"{name:34s} seed {seed}: F1 {r['f1']:.3f} (95% CI {lo:.3f}-{hi:.3f}) P {r['prec']:.3f} R {r['rec']:.3f} "
          f"| recall smishing {r['recall_smishing']:.3f} spam {r['recall_spam']:.3f} no-URL {r['recall_mal_no_url']:.3f}")

In [5]:
# ---- Cell 4: non-contextual baselines (Section 3.1) ----
ytr = tr.label_bin.values
record("majority (always malicious)", None, np.ones(len(va)))

Xl = lambda d: np.log1p(d.n_chars.values).reshape(-1, 1)
m = LogisticRegression(class_weight="balanced").fit(Xl(tr), ytr)
record("length-only LR", None, m.predict_proba(Xl(va))[:, 1])

Xu = lambda d: d.has_url.astype(int).values.reshape(-1, 1)
m = LogisticRegression(class_weight="balanced").fit(Xu(tr), ytr)
record("URL-presence-only LR", None, m.predict_proba(Xu(va))[:, 1])

vw = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
Xt = hstack([vw.fit_transform(tr.text_model), vc.fit_transform(tr.text_model)]).tocsr()
Xv = hstack([vw.transform(va.text_model), vc.transform(va.text_model)]).tocsr()
m = LogisticRegression(class_weight="balanced", max_iter=2000).fit(Xt, ytr)
record("TF-IDF + LR", None, m.predict_proba(Xv)[:, 1])

majority (always malicious)        seed None: F1 0.765 (95% CI 0.732-0.800) P 0.619 R 1.000 | recall smishing 1.000 spam 1.000 no-URL 1.000
length-only LR                     seed None: F1 0.814 (95% CI 0.777-0.851) P 0.811 R 0.818 | recall smishing 0.726 spam 0.851 no-URL 0.250
URL-presence-only LR               seed None: F1 0.913 (95% CI 0.886-0.938) P 0.911 R 0.915 | recall smishing 0.871 spam 0.931 no-URL 0.000
TF-IDF + LR                        seed None: F1 0.964 (95% CI 0.945-0.979) P 0.970 R 0.958 | recall smishing 0.919 spam 0.971 no-URL 0.550


In [6]:
# ---- Cell 5: RoBERTa fine-tuning utilities (plain PyTorch loop, no Trainer-version dependence) ----
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu" and not SMOKE:
    print("WARNING: no GPU. Turn on Settings -> Accelerator, or set SMOKE = True.")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class DS(torch.utils.data.Dataset):
    def __init__(self, enc, y): self.enc, self.y = enc, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i], "label": int(self.y[i])}

def collate(tok):
    def f(batch):
        feats = [{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch]
        out = tok.pad(feats, return_tensors="pt")
        out["labels"] = torch.tensor([b["label"] for b in batch])
        return out
    return f

@torch.no_grad()
def predict(model, loader):
    """Returns P(malicious) and the first-token (<s>) embedding of the last layer."""
    model.eval(); P, E = [], []
    for b in loader:
        b.pop("labels"); b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            out = model(**b, output_hidden_states=True)
        P.append(torch.softmax(out.logits.float(), -1)[:, 1].cpu().numpy())
        E.append(out.hidden_states[-1][:, 0, :].float().cpu().numpy())
    return np.concatenate(P), np.concatenate(E)

def run_finetune(trd, vad, weighted, seed, text_col="text_model"):
    set_seed(seed); tok = make_tok()
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
    model.resize_token_embeddings(len(tok)); model.to(device)
    y_tr = trd.label_bin.values
    w = None
    if weighted:
        cnt = np.bincount(y_tr, minlength=2)
        w = torch.tensor(len(y_tr) / (2.0 * cnt), dtype=torch.float, device=device)   # n / (k * n_c)
    lossf = torch.nn.CrossEntropyLoss(weight=w)
    enc = lambda s: tok(list(s), truncation=True, max_length=MAX_LEN)
    dtr, dva = DS(enc(trd[text_col]), y_tr), DS(enc(vad[text_col]), vad.label_bin.values)
    g = torch.Generator(); g.manual_seed(seed)
    ltr = DataLoader(dtr, batch_size=BATCH, shuffle=True, generator=g, collate_fn=collate(tok))
    lva = DataLoader(dva, batch_size=64, collate_fn=collate(tok))
    ltr_eval = DataLoader(dtr, batch_size=64, collate_fn=collate(tok))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(ltr)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
    best = (-1.0, 0, None)
    for ep in range(EPOCHS):
        model.train()
        for b in ltr:
            y = b.pop("labels").to(device); b = {k: v.to(device) for k, v in b.items()}
            opt.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                logits = model(**b).logits
            loss = lossf(logits.float(), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        pv, _ = predict(model, lva)
        f1 = prf(vad.label_bin.values, (pv >= 0.5).astype(int))["f1"]
        print(f"  epoch {ep + 1}/{EPOCHS}  val F1 {f1:.3f}")
        if f1 > best[0]:
            best = (f1, ep + 1, copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()}))
    model.load_state_dict(best[2])
    print(f"  kept epoch {best[1]} (selected on VALIDATION F1; validation figures are therefore optimistic)")
    return model, tok, best[1], lva, ltr_eval

device: cuda


In [7]:
# ---- Cell 6: runs: (A) class-weighted loss, (B) unweighted + SMOTE head, optional [LINK] ablation ----
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn"], check=True)
    from imblearn.over_sampling import SMOTE

def link_removed(s):
    return s.str.replace("[LINK]", "", regex=False).str.replace(r"\s+", " ", regex=True).str.strip()

for seed in SEEDS:
    print(f"\n=== seed {seed}: A. class-weighted loss ===")
    mdl, tok, ep, lva, ltr_eval = run_finetune(tr, va, True, seed)
    pv, _ = predict(mdl, lva); record("RoBERTa class-weighted", seed, pv)
    if seed == SEEDS[0] and not SMOKE:
        mdl.save_pretrained(f"{OUT}/roberta_weighted_seed{seed}"); tok.save_pretrained(f"{OUT}/roberta_weighted_seed{seed}")
    del mdl; torch.cuda.empty_cache()

    print(f"\n=== seed {seed}: B. unweighted fine-tune, then SMOTE on training embeddings ===")
    mdl, tok, ep, lva, ltr_eval = run_finetune(tr, va, False, seed)
    pv, Ev = predict(mdl, lva); _, Et = predict(mdl, ltr_eval)
    record("RoBERTa unweighted", seed, pv)
    Xb, yb = SMOTE(random_state=seed).fit_resample(Et, tr.label_bin.values)
    sc = StandardScaler().fit(Xb)
    clf = LogisticRegression(max_iter=3000).fit(sc.transform(Xb), yb)
    record("RoBERTa unweighted + SMOTE head", seed, clf.predict_proba(sc.transform(Ev))[:, 1])
    del mdl; torch.cuda.empty_cache()

    if RUN_LINK_ABLATION:
        print(f"\n=== seed {seed}: C. class-weighted, [LINK] removed ===")
        tr2 = tr.assign(text_nl=link_removed(tr.text_model)); va2 = va.assign(text_nl=link_removed(va.text_model))
        mdl, tok, ep, lva, ltr_eval = run_finetune(tr2, va2, True, seed, text_col="text_nl")
        pv, _ = predict(mdl, lva); record("RoBERTa class-weighted, no [LINK]", seed, pv)
        del mdl; torch.cuda.empty_cache()


=== seed 42: A. class-weighted loss ===


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  437MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: jcblaise/roberta-tagalog-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.s

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: jcblaise/roberta-tagalog-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/4  val F1 0.968
  epoch 2/4  val F1 0.971
  epoch 3/4  val F1 0.975
  epoch 4/4  val F1 0.975
  kept epoch 3 (selected on VALIDATION F1; validation figures are therefore optimistic)
RoBERTa class-weighted             seed 44: F1 0.975 (95% CI 0.959-0.987) P 0.971 R 0.979 | recall smishing 0.968 spam 0.983 no-URL 0.800

=== seed 44: B. unweighted fine-tune, then SMOTE on training embeddings ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: jcblaise/roberta-tagalog-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/4  val F1 0.971
  epoch 2/4  val F1 0.973
  epoch 3/4  val F1 0.975
  epoch 4/4  val F1 0.970
  kept epoch 3 (selected on VALIDATION F1; validation figures are therefore optimistic)
RoBERTa unweighted                 seed 44: F1 0.975 (95% CI 0.959-0.987) P 0.971 R 0.979 | recall smishing 0.968 spam 0.983 no-URL 0.800
RoBERTa unweighted + SMOTE head    seed 44: F1 0.966 (95% CI 0.949-0.981) P 0.962 R 0.970 | recall smishing 0.935 spam 0.983 no-URL 0.750


In [8]:
# ---- Cell 7: summary (validation only) ----
res = pd.DataFrame(results)
res.to_csv(f"{OUT}/results_val_runs.csv", index=False)
cols = ["f1", "prec", "rec", "acc", "recall_smishing", "recall_spam", "recall_mal_no_url"]
summary = res.groupby("model", sort=False)[cols].agg(["mean", "std"]).round(3)
print(summary.to_string())
json.dump(dict(model_id=MODEL_ID, max_len=MAX_LEN, epochs=EPOCHS, lr=LR, batch=BATCH, seeds=SEEDS,
               smoke=SMOKE, corpus=os.path.basename(path), n_train=len(tr), n_val=len(va)),
          open(f"{OUT}/config_used.json", "w"), indent=2)
print("\nSaved results_val_runs.csv and config_used.json. The TEST split has not been used.")

                                    f1          prec           rec           acc        recall_smishing        recall_spam        recall_mal_no_url       
                                  mean    std   mean    std   mean    std   mean    std            mean    std        mean    std              mean    std
model                                                                                                                                                     
majority (always malicious)      0.765    NaN  0.619    NaN  1.000    NaN  0.619    NaN           1.000    NaN       1.000    NaN             1.000    NaN
length-only LR                   0.814    NaN  0.811    NaN  0.818    NaN  0.769    NaN           0.726    NaN       0.851    NaN             0.250    NaN
URL-presence-only LR             0.913    NaN  0.911    NaN  0.915    NaN  0.892    NaN           0.871    NaN       0.931    NaN             0.000    NaN
TF-IDF + LR                      0.964    NaN  0.970    NaN  0.958    

## What to send back
1. The **Cell 2** coverage table and the **median / p95 / p99** line.
2. The **Cell 4** baseline lines.
3. The **Cell 7** summary table (mean and std over seeds).
4. Any error message, complete, from any cell.

**Do not** report these validation numbers as final results: models were selected on validation, and the test split is untouched. Final numbers come from one run on the test split after the configuration is frozen.